# Encoder screen — TIER 1: frozen linear probe (fold0)

**Goal:** a perfectly uniform, reviewer-defensible broad comparison of backbones, used to **shortlist** the top
encoders. Both vision and text encoders are **fully frozen**; only the projection heads (+ temperature) train.
Identical treatment for every backbone -> no per-architecture unfreeze confound.

**Fixed (deliberately plain):** loss = `L1_ce` (standard cross-entropy, no class-balancing / no alpha-beta),
similarity = `cosine`. text fixed = `mpnet` during the vision stage.

- **Stage A (Cell 6):** 9 vision backbones (frozen) -> rank -> top vision(s).
- **Stage B (Cell 7):** 5 text encoders (frozen) on the best vision -> rank.
- **Cell 8:** shortlist summary.

Numbers here are LOWER than a fine-tuned model (frozen features + tiny head) and only used to **shortlist** the
top ~2-3 vision + top ~2 text. **Tier 2** then fine-tunes (partial unfreeze) only the shortlist and makes the
final pick under the real regime.

Resume-safe. ~14 runs, head-only training -> fast (~1-1.5 h). Set `QUICK_TEST=True` first to verify wiring/GPU.

In [1]:
# Cell 1. Config — TIER 1 FROZEN LINEAR-PROBE screen (both encoders frozen, plain CE, cosine)
import warnings, os, logging, gc, time, re
from contextlib import nullcontext
warnings.filterwarnings("ignore"); logging.getLogger("transformers").setLevel(logging.ERROR)
# --- data-disk cache + CN mirror (MUST be set before transformers import / any download) ---
os.environ["HF_HOME"]="/root/autodl-tmp/hf_cache"
os.environ["HF_ENDPOINT"]="https://hf-mirror.com"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"]="1"; os.environ["HF_HUB_DISABLE_TELEMETRY"]="1"
from pathlib import Path
Path("/root/autodl-tmp/hf_cache").mkdir(parents=True, exist_ok=True)
LOCAL_ONLY=False

SPLIT_ROOT=Path("splits_clip/cv5"); CLASSES_CSV=Path("splits_clip/classes.csv")
IMG_ROOT=Path("."); ABL_FOLD=0
OUTDIR=Path("clip_encoder_screen_frozen"); OUTDIR.mkdir(exist_ok=True)

# === FIXED screening config (Tier 1: uniform & deliberately plain) ===
FIXED_LOSS = "L1_ce"     # standard cross-entropy (no balancing). plain & widely accepted.
SIM        = "cosine"
DEFAULT_TEXT_ID, DEFAULT_TEXT = "sentence-transformers/all-mpnet-base-v2", "mpnet"
# === BOTH ENCODERS FULLY FROZEN -> identical treatment for every backbone ===
FREEZE_VIS, FREEZE_TXT = 1.0, 1.0
# ====================================================================

TEXT_ID=DEFAULT_TEXT_ID
PROJ_DIM=512
EPOCHS, MIN_EPOCHS, PATIENCE = 100, 40, 30   # head-only -> converges fast
BATCH, OOM_BATCH = 16, 4
LR_HEAD, LR_ENC, MAX_TOK, GRAD_CLIP, SEED = 1e-3, 1e-5, 32, 1.0, 42
AUGMENT=True; USE_SAMPLER=True

VISIONS = {
    "resnet50":        "microsoft/resnet-50",
    "vit_base":        "google/vit-base-patch16-224",
    "swin_base":       "microsoft/swin-base-patch4-window7-224",
    "swinv2_base":     "microsoft/swinv2-base-patch4-window8-256",
    "beit_base":       "microsoft/beit-base-patch16-224",
    "dinov2_base":     "facebook/dinov2-base",
    "convnextv2_base": "facebook/convnextv2-base-22k-224",
    "clip_base":       "openai/clip-vit-base-patch32",
    "siglip_base":     "google/siglip-base-patch16-224",
}
TEXTS = {
    "mpnet":        "sentence-transformers/all-mpnet-base-v2",
    "bert_base":    "bert-base-uncased",
    "roberta_base": "roberta-base",
    "deberta_v3":   "microsoft/deberta-v3-base",     # needs sentencepiece + protobuf
    "distilbert":   "distilbert-base-uncased",
}
VISIONS_FULL=dict(VISIONS)

QUICK_TEST = False   # True: 2 vision x 2 text x few epochs to verify wiring + GPU. set False for the real run.
if QUICK_TEST:
    VISIONS={k:VISIONS[k] for k in ["swinv2_base","dinov2_base"]}
    TEXTS  ={k:TEXTS[k]   for k in ["mpnet","bert_base"]}
    EPOCHS, MIN_EPOCHS, PATIENCE = 8, 3, 4

print(f"OUTDIR={OUTDIR} | fold {ABL_FOLD} | TIER-1 FROZEN | loss={FIXED_LOSS} sim={SIM}")
print(f"freeze: vision={FREEZE_VIS} text={FREEZE_TXT}  (1.0 = fully frozen, heads only)")
print(f"STAGE A: {len(VISIONS)} vision (frozen) | STAGE B: {len(TEXTS)} text (frozen) | QUICK_TEST={QUICK_TEST}")

OUTDIR=clip_encoder_screen_frozen | fold 0 | TIER-1 FROZEN | loss=L1_ce sim=cosine
freeze: vision=1.0 text=1.0  (1.0 = fully frozen, heads only)
STAGE A: 9 vision (frozen) | STAGE B: 5 text (frozen) | QUICK_TEST=False


In [2]:
# Cell 2. Imports + data + dataset/loader (same as Method 1)
import numpy as np, pandas as pd, random, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
from torchvision import transforms
from transformers import AutoModel, AutoImageProcessor, AutoTokenizer
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from collections import deque
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
set_seed(SEED)
DEV="cuda" if torch.cuda.is_available() else "cpu"; BF16=(DEV=="cuda" and torch.cuda.is_bf16_supported())
print("device:",DEV,"| bf16:",BF16)

CLASSES=list(pd.read_csv(CLASSES_CSV)["caption"]); CIDX={c:i for i,c in enumerate(CLASSES)}; NCLS=len(CLASSES)
def load_fold(k):
    out={}
    for nm in ["train","val","test"]:
        df=pd.read_csv(SPLIT_ROOT/f"fold{k}"/f"{nm}.csv")
        df["label"]=df["caption_norm"].map(CIDX); out[nm]=df.reset_index(drop=True)
    return out
def metrics(true,pred):
    true=list(true); pred=list(pred)
    labs=sorted(set(true))   # 只在测试集中实际出现的类上算 macro(标准做法,不被缺席类拉低)
    acc=accuracy_score(true,pred)
    pma,rma,fma,_=precision_recall_fscore_support(true,pred,labels=labs,average="macro",zero_division=0)
    pw,rw,fw,_=precision_recall_fscore_support(true,pred,labels=labs,average="weighted",zero_division=0)
    return {"acc":acc,"macroF1":fma,"macroP":pma,"macroR":rma,"weightedF1":fw}

AUG=transforms.Compose([transforms.RandomHorizontalFlip(),
                        transforms.ColorJitter(0.2,0.2,0.2,0.02),
                        transforms.RandomRotation(8)])
class ImgDS(Dataset):
    def __init__(self,df,proc,train):
        self.df=df; self.proc=proc; self.train=train and AUGMENT
        self.y=torch.tensor(df["label"].values)
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        img=Image.open(IMG_ROOT/self.df.iloc[i]["image"]).convert("RGB")
        if self.train: img=AUG(img)
        pix=self.proc(images=img,return_tensors="pt")["pixel_values"][0]
        return pix, self.y[i], i
def vis_pool(enc,pix):
    m=getattr(enc,"vision_model",enc); h=m(pixel_values=pix).last_hidden_state
    if h.dim()==4: h=h.flatten(2).transpose(1,2)
    return h.mean(1)
def make_loader(df,proc,train):
    ds=ImgDS(df,proc,train)
    if train and USE_SAMPLER:
        cnt=df["label"].value_counts(); w=df["label"].map(lambda l:1.0/cnt[l]).values
        sm=WeightedRandomSampler(torch.as_tensor(w,dtype=torch.double),num_samples=len(df),replacement=True)
        return DataLoader(ds,batch_size=BATCH,sampler=sm,num_workers=0)
    return DataLoader(ds,batch_size=BATCH,shuffle=train,num_workers=0)
print("data ready, NCLS=",NCLS)


device: cuda | bf16: True
data ready, NCLS= 30


In [3]:
# Cell 3. Loss implementations -- the heart of this notebook
class LossFunctions:
    # All losses operate on: logits [B, NCLS] from cosine*scale, labels [B], 
    # image_emb [B, D] normalized, text_emb [NCLS, D] normalized, batch_pos [B, D] = text_emb[labels]
    # class_counts [NCLS] long-tail prior info

    @staticmethod
    def l0_infonce(logits, labels, ie, te, class_counts):
        # symmetric InfoNCE on (image -> text) and (text -> image)
        tpos = te[labels]
        scale = logits.max().item() / max((ie @ te.t())[0,0].item(), 1e-8) if False else None
        # Reconstruct from scratch for clarity
        return F.cross_entropy(logits, labels)  # InfoNCE here = CE over class-prompts when using same logits matrix
        # Note: the real InfoNCE you trained with is symmetric (img->txt + txt->img),
        # but for fold0 ablation we use single-direction CE on the same cosine logits matrix.

    @staticmethod  
    def l0_infonce_symmetric(logits, labels, ie, te, class_counts):
        # True symmetric InfoNCE: image to its positive text + positive text to image
        tpos = te[labels]   # [B, D]
        lc_i2t = logits[:, labels]  # but this is wrong shape; use direct dot
        # Compute pairwise sim between batch_img and batch_pos
        s = ie.size(0)
        lc = (logits[range(s)].t()[labels].t()) if False else None
        # Simplest: standard image-text contrastive with batch-as-negatives
        # logits_i2t[i,j] = sim(img_i, txt_pos_j) where txt_pos_j = text_emb[labels[j]]
        scale = (logits.max() / (ie@te.t()).max()).item() if (ie@te.t()).max()>0 else 100.0
        lc = scale * (ie @ tpos.t())
        tgt = torch.arange(ie.size(0), device=ie.device)
        return 0.5 * (F.cross_entropy(lc, tgt) + F.cross_entropy(lc.t(), tgt))

    @staticmethod
    def l1_ce(logits, labels, ie, te, class_counts):
        return F.cross_entropy(logits, labels)

    @staticmethod
    def l2_balanced_softmax(logits, labels, ie, te, class_counts):
        adj = torch.log(class_counts.float() + 1e-8).to(logits.device).unsqueeze(0)
        return F.cross_entropy(logits + adj, labels)

    @staticmethod
    def l3_logit_adjust(logits, labels, ie, te, class_counts, tau=1.0):
        prior = (class_counts.float() / class_counts.sum()).to(logits.device)
        adj = (tau * torch.log(prior + 1e-8)).unsqueeze(0)
        return F.cross_entropy(logits + adj, labels)

    @staticmethod
    def l4_focal(logits, labels, ie, te, class_counts, gamma=2.0):
        ce = F.cross_entropy(logits, labels, reduction="none")
        p = (-ce).exp()  # p_y = exp(-CE)
        return ((1-p)**gamma * ce).mean()

    @staticmethod
    def l5_cb_loss(logits, labels, ie, te, class_counts, beta=0.99):
        eff_num = (1.0 - beta**class_counts.float()) / (1.0 - beta + 1e-8)
        weights = (1.0 / (eff_num + 1e-8))
        weights = weights / weights.sum() * NCLS  # normalize so mean weight ~ 1
        return F.cross_entropy(logits, labels, weight=weights.to(logits.device))

    @staticmethod
    def l6_ldam(logits, labels, ie, te, class_counts, max_m=0.5, s=30.0):
        m_list = 1.0 / torch.sqrt(torch.sqrt(class_counts.float() + 1e-8))
        m_list = m_list * (max_m / m_list.max())
        m_list = m_list.to(logits.device)
        index = torch.zeros_like(logits, dtype=torch.bool)
        index.scatter_(1, labels.unsqueeze(1), True)
        margin_batch = m_list[labels].unsqueeze(1)
        logits_m = logits - index.float() * margin_batch
        return F.cross_entropy(logits_m * s / logits.max().clamp(min=1.0), labels)

    @staticmethod
    def l7_supcon(logits, labels, ie, te, class_counts, temp=0.07):
        # Supervised contrastive on image features: same-class samples in batch are positives
        z = F.normalize(ie, dim=-1)
        sim = z @ z.t() / temp  # [B, B]
        # mask diagonal
        mask_diag = torch.eye(z.size(0), device=z.device, dtype=torch.bool)
        sim.masked_fill_(mask_diag, -1e9)
        # positive mask: same label, not self
        labels_eq = labels.unsqueeze(0) == labels.unsqueeze(1)
        pos_mask = labels_eq & ~mask_diag
        # log_prob
        logits_max, _ = sim.max(dim=1, keepdim=True)
        sim_norm = sim - logits_max.detach()
        exp_sim = sim_norm.exp()
        log_prob = sim_norm - torch.log(exp_sim.sum(1, keepdim=True) + 1e-12)
        # mean log_prob over positives (skip samples with no positives)
        n_pos = pos_mask.sum(1)
        valid = n_pos > 0
        if valid.sum() == 0: 
            return F.cross_entropy(logits, labels)  # fallback
        mean_log_prob = (pos_mask.float() * log_prob).sum(1)[valid] / n_pos[valid].float()
        loss_supcon = -mean_log_prob.mean()
        # Add the cosine-CE for classifier supervision (otherwise model has no class anchor)
        return 0.5 * loss_supcon + 0.5 * F.cross_entropy(logits, labels)

    @staticmethod
    def l8_arcface(logits, labels, ie, te, class_counts, m=0.5, s=30.0):
        # logits are scale * cosine; recover cosine
        cos_theta = logits / max(logits.abs().max().item(), 1.0)
        cos_theta = cos_theta.clamp(-1+1e-7, 1-1e-7)
        theta = cos_theta.acos()
        # Only apply margin to ground-truth class
        index = torch.zeros_like(logits, dtype=torch.bool)
        index.scatter_(1, labels.unsqueeze(1), True)
        theta_m = theta + index.float() * m
        cos_theta_m = theta_m.cos()
        out = s * cos_theta_m
        return F.cross_entropy(out, labels)

    @staticmethod
    def l9_cosface(logits, labels, ie, te, class_counts, m=0.4, s=30.0):
        cos_theta = logits / max(logits.abs().max().item(), 1.0)
        cos_theta = cos_theta.clamp(-1+1e-7, 1-1e-7)
        index = torch.zeros_like(logits, dtype=torch.bool)
        index.scatter_(1, labels.unsqueeze(1), True)
        cos_theta_m = cos_theta - index.float() * m
        out = s * cos_theta_m
        return F.cross_entropy(out, labels)

LOSS_FN = {
    "L0_infonce":            LossFunctions.l0_infonce_symmetric,
    "L1_ce":                 LossFunctions.l1_ce,
    "L2_balanced_softmax":   LossFunctions.l2_balanced_softmax,
    "L3_logit_adjust_t10":   LossFunctions.l3_logit_adjust,
    "L4_focal_g2":           LossFunctions.l4_focal,
    "L5_cb_loss_b099":       LossFunctions.l5_cb_loss,
    "L6_ldam":               LossFunctions.l6_ldam,
    "L7_supcon":             LossFunctions.l7_supcon,
    "L8_arcface_m05":        LossFunctions.l8_arcface,
    "L9_cosface_m04":        LossFunctions.l9_cosface,
}
print("loss functions:", list(LOSS_FN.keys()))


loss functions: ['L0_infonce', 'L1_ce', 'L2_balanced_softmax', 'L3_logit_adjust_t10', 'L4_focal_g2', 'L5_cb_loss_b099', 'L6_ldam', 'L7_supcon', 'L8_arcface_m05', 'L9_cosface_m04']


In [4]:
# Cell 4. Model (same architecture, no logit_adjust param -- losses see raw cosine logits)
def partial_unfreeze(model, freeze_frac):
    for p in model.parameters(): p.requires_grad=False
    if freeze_frac>=0.999:
        return 0   # FROZEN: encoder fully frozen, only projection heads train (linear probe)
    names=[n for n,_ in model.named_parameters()]
    li=[int(m.group(1)) for n in names for m in [re.search(r"encoder\.layer\.(\d+)\.",n)] if m]
    si=[int(m.group(1)) for n in names for m in [re.search(r"encoder\.layers\.(\d+)\.",n)] if m]
    ntr=0
    if li:
        cut=int(round(max(li)*freeze_frac))
        for n,p in model.named_parameters():
            m=re.search(r"encoder\.layer\.(\d+)\.",n)
            if (m and int(m.group(1))>=cut) or any(k in n.lower() for k in ["layernorm","pooler"]): p.requires_grad=True; ntr+=p.numel()
    elif si:
        cut=int(round((max(si)+1)*freeze_frac))
        for n,p in model.named_parameters():
            m=re.search(r"encoder\.layers\.(\d+)\.",n)
            if (m and int(m.group(1))>=cut) or "layernorm" in n.lower(): p.requires_grad=True; ntr+=p.numel()
    else:
        for p in model.parameters(): p.requires_grad=True; ntr+=p.numel()
    return ntr

class CLIPRetriever(nn.Module):
    def __init__(self, vis_id, txt_id, proc, D=PROJ_DIM):
        super().__init__()
        self.venc=AutoModel.from_pretrained(vis_id,torch_dtype=torch.float32,local_files_only=LOCAL_ONLY)
        self.tenc=AutoModel.from_pretrained(txt_id,torch_dtype=torch.float32,local_files_only=LOCAL_ONLY)
        with torch.no_grad():
            dummy=proc(images=Image.new("RGB",(256,256)),return_tensors="pt")["pixel_values"]
            dv=vis_pool(self.venc,dummy).shape[-1]
        dt=self.tenc.config.hidden_size
        self.ip=nn.Sequential(nn.Linear(dv,D),nn.GELU(),nn.Linear(D,D))
        self.tp=nn.Sequential(nn.Linear(dt,D),nn.GELU(),nn.Linear(D,D))
        self.scale=nn.Parameter(torch.tensor(float(np.log(1/0.07))))
        nv=partial_unfreeze(self.venc,FREEZE_VIS); nt=partial_unfreeze(self.tenc,FREEZE_TXT)
        print(f"   unfrozen: vision {nv:,} | text {nt:,}")
    def enc_text(self, ids, mask):
        o=self.tenc(input_ids=ids,attention_mask=mask).last_hidden_state
        m=mask.unsqueeze(-1).float(); return self.tp((o*m).sum(1)/m.sum(1).clamp(min=1))
    def enc_img(self, pix): return self.ip(vis_pool(self.venc,pix))
    def forward_features(self, pix, tids, tmask):
        ie=self.enc_img(pix); te=self.enc_text(tids,tmask)
        s=self.scale.exp().clamp(max=100.0)
        ien=F.normalize(ie,dim=-1); ten=F.normalize(te,dim=-1)
        logits = s * ien @ ten.t()
        return logits, ien, ten
    @torch.no_grad()
    def predict(self, pix, tids, tmask):
        lg, _, _ = self.forward_features(pix, tids, tmask)
        return lg
print("model class ready")


model class ready


In [5]:
# Cell 5. Single-loss training function
def train_run(loss_name, vis_id, txt_id, proc, tok, txt_ids, txt_mask, fold, batch):
    set_seed(SEED)   # 每个 run 从同一随机起点:在建模型/读数据之前重置 -> init+数据顺序对所有配置一致
    F_=load_fold(fold)
    model=CLIPRetriever(vis_id, txt_id, proc).to(DEV)
    head=[p for n,p in model.named_parameters() if p.requires_grad and ("ip." in n or "tp." in n or n=="scale")]
    enc=[p for n,p in model.named_parameters() if p.requires_grad and not("ip." in n or "tp." in n or n=="scale")]
    grps=[{"params":head,"lr":LR_HEAD}]+([{"params":enc,"lr":LR_ENC}] if enc else [])
    opt=torch.optim.AdamW(grps,weight_decay=0.01)

    cnt=F_["train"]["label"].value_counts()
    class_counts=torch.tensor([cnt.get(i,1) for i in range(NCLS)], dtype=torch.long, device=DEV)
    print(f"   class_counts: min={class_counts.min().item()} max={class_counts.max().item()}")

    loss_fn = LOSS_FN[loss_name]
    dl=make_loader(F_["train"],proc,True)
    TI,TM=txt_ids.to(DEV),txt_mask.to(DEV)
    def ev(split):
        model.eval(); P=[]; T=[]
        for pix,y,_ in DataLoader(ImgDS(F_[split],proc,False),batch_size=batch):
            lg=model.predict(pix.to(DEV),TI,TM); P+=lg.argmax(1).cpu().tolist(); T+=y.tolist()
        return T,P

    win=deque(maxlen=3); best=-1; bstate=None; bad=0; bep=-1
    nan_streak=0
    for ep in range(EPOCHS):
        model.train(); run=0; st=0
        for pix,y,_ in dl:
            opt.zero_grad()
            ctx=torch.autocast("cuda",dtype=torch.bfloat16) if BF16 else nullcontext()
            with ctx:
                logits, ie, te = model.forward_features(pix.to(DEV),TI,TM)
                loss = loss_fn(logits, y.to(DEV), ie, te, class_counts)
            if not torch.isfinite(loss): 
                nan_streak += 1
                if nan_streak > 50: 
                    print(f"   ABORT: loss non-finite for 50 batches")
                    return None
                continue
            nan_streak = 0
            loss.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],GRAD_CLIP)
            opt.step()
            with torch.no_grad(): model.scale.clamp_(max=float(np.log(100.0)))
            run+=loss.item(); st+=1
        T,P=ev("val"); m=metrics(T,P); sc=m["acc"]+m["macroF1"]; win.append(sc); sm=sum(win)/len(win)
        if sm>best: best=sm; bep=ep; bad=0; bstate={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        elif ep>=MIN_EPOCHS: bad+=1
        if ep%5==0 or ep==EPOCHS-1: print(f"   ep{ep:03d} loss={run/max(1,st):.3f} val_acc={m['acc']:.3f} val_mF1={m['macroF1']:.3f}")
        if ep>=MIN_EPOCHS and bad>=PATIENCE: print(f"   early stop @ep{ep} (best @ep{bep})"); break

    if bstate: model.load_state_dict(bstate)
    T,P=ev("test"); fm=metrics(T,P)
    del model,opt,dl; gc.collect(); torch.cuda.empty_cache() if DEV=="cuda" else None
    return fm


In [6]:
# Cell 6. STAGE A — vision screen (FROZEN). 只跳过已成功(OK)的;缺失/失败的(swinv2、clip)重新补跑。
from transformers import AutoTokenizer, AutoImageProcessor
print(f">>> GPU OK: {torch.cuda.get_device_name(0)} | bf16={BF16}") if DEV=="cuda" else print(">>> WARNING: NO GPU")

def text_tensors(txt_id):
    tok=AutoTokenizer.from_pretrained(txt_id, local_files_only=LOCAL_ONLY)
    e=tok(CLASSES,padding="max_length",truncation=True,max_length=MAX_TOK,return_tensors="pt")
    return tok, e["input_ids"], e["attention_mask"]

def run_combo(vis_id, txt_id):
    proc=AutoImageProcessor.from_pretrained(vis_id, local_files_only=LOCAL_ONLY)
    tok,TI,TM=text_tensors(txt_id)
    try:
        return train_run(FIXED_LOSS, vis_id, txt_id, proc, tok, TI, TM, ABL_FOLD, BATCH)
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            torch.cuda.empty_cache(); gc.collect(); print(f"   OOM -> retry batch {OOM_BATCH}")
            return train_run(FIXED_LOSS, vis_id, txt_id, proc, tok, TI, TM, ABL_FOLD, OOM_BATCH)
        raise

def append_row(csv, base, fm, t0):
    if fm is None:
        row={**base,"status":"FAILED_NAN","minutes":round((time.time()-t0)/60,1)}
        for k in ["acc","macroF1","macroP","macroR","weightedF1"]: row[k]=float("nan")
    else:
        row={**base,"status":"OK","minutes":round((time.time()-t0)/60,1),
             **{k:round(v,4) for k,v in fm.items()}}
        print(f"   DONE: acc={fm['acc']:.4f} macroF1={fm['macroF1']:.4f}")
    pd.DataFrame([row]).to_csv(csv,mode="a",header=not csv.exists(),index=False)

VIS_CSV=OUTDIR/"vision_screen_frozen.csv"
# --- 只保留成功(OK)的记录;失败/残缺的清掉,以便干净地重跑(避免重复行)---
if VIS_CSV.exists():
    _df=pd.read_csv(VIS_CSV)
    _ok=_df[_df["status"]=="OK"].drop_duplicates(subset=["vision"], keep="last")
    _ok.to_csv(VIS_CSV, index=False)
    done=set(_ok["vision"].tolist())
else:
    done=set()
todo=[v for v in VISIONS if v not in done]
print(f"\n{'#'*64}\n# STAGE A (FROZEN): 已成功(跳过)={sorted(done)}\n# 本次将跑={todo}\n{'#'*64}")

for vname in todo:
    vid=VISIONS[vname]
    print("="*60); print(f"STAGE A vision = {vname}")
    t0=time.time(); base={"stage":"A","vision":vname,"text":DEFAULT_TEXT,"loss":FIXED_LOSS,"sim":SIM,"frozen":True}
    try:
        fm=run_combo(vid, DEFAULT_TEXT_ID); append_row(VIS_CSV, base, fm, t0)
    except Exception as e:
        print(f"   FAILED: {type(e).__name__}: {str(e)[:200]}")
        b={**base,"status":f"FAILED_{type(e).__name__}","minutes":round((time.time()-t0)/60,1)}
        for k in ["acc","macroF1","macroP","macroR","weightedF1"]: b[k]=float("nan")
        pd.DataFrame([b]).to_csv(VIS_CSV,mode="a",header=not VIS_CSV.exists(),index=False)
print("\n##### STAGE A done #####")

>>> GPU OK: NVIDIA GeForce RTX 4090 | bf16=True

################################################################
# STAGE A (FROZEN): 已成功(跳过)=['beit_base', 'clip_base', 'convnextv2_base', 'dinov2_base', 'resnet50', 'siglip_base', 'swin_base', 'swinv2_base', 'vit_base']
# 本次将跑=[]
################################################################

##### STAGE A done #####


In [7]:
# Cell 7a. STAGE A 排名 —— 只看结果,决定保留几个 vision(不训练)
vis_df=pd.read_csv(OUTDIR/"vision_screen_frozen.csv")
vis_ok=vis_df[vis_df["status"]=="OK"].sort_values("acc",ascending=False).reset_index(drop=True)
assert len(vis_ok), "No successful Stage A runs — run Cell 6 first."
top=vis_ok.iloc[0]["acc"]
import pandas as pd; pd.set_option("display.float_format", lambda x:f"{x:.4f}")
print("="*74); print("Stage A vision ranking (FROZEN probe, text=mpnet, CE, cosine), by acc"); print("="*74)
print(vis_ok[["vision","acc","macroF1","macroP","macroR","weightedF1","minutes"]].to_string(index=False))
print("\n与第1名的差距:")
for i,r in vis_ok.iterrows():
    print(f"  {i+1:>2}. {r['vision']:<16} acc={r['acc']:.4f}  gap={top-r['acc']:+.4f}")
failed=vis_df[vis_df["status"]!="OK"]["vision"].unique().tolist()
if failed: print(f"\n(未成功/失败,未计入排名: {failed})")
print("\n>>> 看完排名后,到 Cell 7b 设 N_VISION_KEEP 或手动写 KEEP,再跑 Stage B。")

Stage A vision ranking (FROZEN probe, text=mpnet, CE, cosine), by acc
         vision    acc  macroF1  macroP  macroR  weightedF1  minutes
    swinv2_base 0.8862   0.7519  0.7814  0.7714      0.8761   8.4000
    dinov2_base 0.8374   0.7095  0.7304  0.7426      0.8273   8.1000
      beit_base 0.8211   0.6563  0.6809  0.6683      0.8089   6.7000
       resnet50 0.8130   0.6583  0.6785  0.6815      0.7851   7.9000
       vit_base 0.8130   0.6820  0.7062  0.7392      0.8053   6.7000
      swin_base 0.8049   0.6622  0.6689  0.7050      0.7888   8.3000
convnextv2_base 0.7805   0.6337  0.6618  0.6935      0.7751   7.1000
    siglip_base 0.7805   0.6544  0.6694  0.7049      0.7578   6.7000
      clip_base 0.7642   0.6372  0.6645  0.6381      0.7556  10.2000

与第1名的差距:
   1. swinv2_base      acc=0.8862  gap=+0.0000
   2. dinov2_base      acc=0.8374  gap=+0.0488
   3. beit_base        acc=0.8211  gap=+0.0651
   4. resnet50         acc=0.8130  gap=+0.0732
   5. vit_base         acc=0.8130  gap=+0.

In [8]:
# Cell 7b. STAGE B — text screen (FROZEN) on chosen visions. Resume-safe (keyed by vision+text).
N_VISION_KEEP = 1     # 自动取前 N 个 vision
KEEP = None           # 想手动指定就写,如 KEEP=["swinv2_base","beit_base"];留 None 则按 N_VISION_KEEP 自动取

vis_df=pd.read_csv(OUTDIR/"vision_screen_frozen.csv")
vis_ok=vis_df[vis_df["status"]=="OK"].sort_values("acc",ascending=False).reset_index(drop=True)
if KEEP is None:
    KEEP=list(vis_ok["vision"].head(N_VISION_KEEP))
print(f">>> 进入 Stage B 的视觉: {KEEP}")

TXT_CSV=OUTDIR/"text_screen_frozen.csv"
done=set()
if TXT_CSV.exists():
    d=pd.read_csv(TXT_CSV); done={(r["vision"],r["text"]) for _,r in d.iterrows() if r.get("status")=="OK"}
print(f"\n{'#'*64}\n# STAGE B (FROZEN): {len(KEEP)} vision x {len(TEXTS)} text | done(OK)={len(done)} pairs\n{'#'*64}")
for vname in KEEP:
    vid=VISIONS_FULL[vname]
    for tname,tid in TEXTS.items():
        if (vname,tname) in done: print(f"[SKIP] {vname} x {tname}"); continue
        print("="*60); print(f"STAGE B  vision={vname}  text={tname}")
        t0=time.time(); base={"stage":"B","vision":vname,"text":tname,"loss":FIXED_LOSS,"sim":SIM,"frozen":True}
        try:
            fm=run_combo(vid, tid); append_row(TXT_CSV, base, fm, t0)
        except Exception as e:
            print(f"   FAILED: {type(e).__name__}: {str(e)[:200]}")
            b={**base,"status":f"FAILED_{type(e).__name__}","minutes":round((time.time()-t0)/60,1)}
            for k in ["acc","macroF1","macroP","macroR","weightedF1"]: b[k]=float("nan")
            pd.DataFrame([b]).to_csv(TXT_CSV,mode="a",header=not TXT_CSV.exists(),index=False)
print("\n##### STAGE B done #####")

>>> 进入 Stage B 的视觉: ['swinv2_base']

################################################################
# STAGE B (FROZEN): 1 vision x 5 text | done(OK)=5 pairs
################################################################
[SKIP] swinv2_base x mpnet
[SKIP] swinv2_base x bert_base
[SKIP] swinv2_base x roberta_base
[SKIP] swinv2_base x deberta_v3
[SKIP] swinv2_base x distilbert

##### STAGE B done #####


In [9]:
# Cell 8. Shortlist summary (Tier 1)
import pandas as pd; pd.set_option("display.float_format", lambda x:f"{x:.4f}")
v=pd.read_csv(OUTDIR/"vision_screen_frozen.csv"); v=v[v["status"]=="OK"].sort_values("acc",ascending=False)
print("="*78); print(f"TIER 1 — vision (FROZEN, text={DEFAULT_TEXT}, {FIXED_LOSS}, {SIM}), by acc"); print("="*78)
print(v[["vision","acc","macroF1","macroP","macroR","weightedF1","minutes"]].to_string(index=False))

tp=OUTDIR/"text_screen_frozen.csv"
if tp.exists():
    t=pd.read_csv(tp); t=t[t["status"]=="OK"].sort_values(["vision","acc"],ascending=[True,False])
    print("\n"+"="*78); print(f"TIER 1 — text screen,每个保留视觉下的排名 ({FIXED_LOSS}, {SIM})"); print("="*78)
    print(t[["vision","text","acc","macroF1","macroP","macroR","weightedF1"]].to_string(index=False))
    best=t.sort_values("acc",ascending=False).iloc[0]
    print("\n"+"="*78); print("##### 最优 (vision, text) 组合(冻结探针) #####")
    print(f"  vision = {best['vision']}   text = {best['text']}   macroF1={best['macroF1']:.4f}  acc={best['acc']:.4f}")
    print(f"  -> 下一步 Tier 2:对入围组合做部分解冻充分训练,定终选,再扫 loss x similarity,再 5 折。")
    print("="*78)
else:
    print("\n(Stage B 还没跑 — 先跑 Cell 7。)")

TIER 1 — vision (FROZEN, text=mpnet, L1_ce, cosine), by acc
         vision    acc  macroF1  macroP  macroR  weightedF1  minutes
    swinv2_base 0.8862   0.7519  0.7814  0.7714      0.8761   8.4000
    dinov2_base 0.8374   0.7095  0.7304  0.7426      0.8273   8.1000
      beit_base 0.8211   0.6563  0.6809  0.6683      0.8089   6.7000
       resnet50 0.8130   0.6583  0.6785  0.6815      0.7851   7.9000
       vit_base 0.8130   0.6820  0.7062  0.7392      0.8053   6.7000
      swin_base 0.8049   0.6622  0.6689  0.7050      0.7888   8.3000
convnextv2_base 0.7805   0.6337  0.6618  0.6935      0.7751   7.1000
    siglip_base 0.7805   0.6544  0.6694  0.7049      0.7578   6.7000
      clip_base 0.7642   0.6372  0.6645  0.6381      0.7556  10.2000

TIER 1 — text screen,每个保留视觉下的排名 (L1_ce, cosine)
     vision         text    acc  macroF1  macroP  macroR  weightedF1
swinv2_base        mpnet 0.8862   0.7519  0.7814  0.7714      0.8761
swinv2_base   distilbert 0.8699   0.7399  0.7708  0.7547      0

In [10]:
# Tier-1 续:冻结探针 + L1_ce 下筛 similarity(swinv2+mpnet);cosine 复用,只跑 dot/euclidean
import time, pandas as pd, torch, torch.nn.functional as F
pd.set_option("display.float_format", lambda x:f"{x:.4f}")
def _ff_sim(self, pix, tids, tmask):
    ie=self.enc_img(pix); te=self.enc_text(tids,tmask); s=self.scale.exp().clamp(max=100.0)
    if SIM=="cosine":
        ien=F.normalize(ie,dim=-1); ten=F.normalize(te,dim=-1); return s*ien@ten.t(), ien, ten
    elif SIM=="dot":   return s*(ie@te.t()), ie, te
    elif SIM=="euclidean":
        d=torch.cdist(ie.float(), te.float(), p=2); return -s*d, ie, te
    else: raise ValueError(f"unknown SIM={SIM}")
CLIPRetriever.forward_features=_ff_sim
print("forward 支持 cosine / dot / euclidean")

VIS_ID="microsoft/swinv2-base-patch4-window8-256"; TXT_ID2="sentence-transformers/all-mpnet-base-v2"
FIXED_LOSS="L1_ce"; FREEZE_VIS, FREEZE_TXT = 1.0, 1.0   # 冻结探针,与 encoder 筛选一致
SC=OUTDIR/"sim_frozen_swinv2_mpnet.csv"
done=set()
if SC.exists(): _d=pd.read_csv(SC); done=set(_d[_d["status"]=="OK"]["sim"])
# cosine 复用 encoder 筛选 swinv2+mpnet 那行(frozen+L1_ce+cosine,完全同配置)
if "cosine" not in done and (OUTDIR/"text_screen_frozen.csv").exists():
    vt=pd.read_csv(OUTDIR/"text_screen_frozen.csv")
    fz=vt[(vt["vision"]=="swinv2_base")&(vt["text"]=="mpnet")&(vt["status"]=="OK")]
    if len(fz):
        r0=fz.iloc[0]
        row={"sim":"cosine","vision":"swinv2_base","text":"mpnet","loss":"L1_ce","freeze":1.0,
             "status":"OK","minutes":0.0,"acc":r0["acc"],"macroF1":r0["macroF1"],
             "macroP":r0["macroP"],"macroR":r0["macroR"],"weightedF1":r0["weightedF1"]}
        pd.DataFrame([row]).to_csv(SC,mode="a",header=not SC.exists(),index=False)
        done.add("cosine"); print(f">>> cosine 复用: acc={r0['acc']:.4f}")
for sn in ["cosine","dot","euclidean"]:
    if sn in done: print(f"[SKIP] {sn}"); continue
    SIM=sn; print("="*60); print(f"SIM={sn} (frozen, L1_ce)")
    t0=time.time(); base={"sim":sn,"vision":"swinv2_base","text":"mpnet","loss":"L1_ce","freeze":1.0}
    try: fm=run_combo(VIS_ID,TXT_ID2); append_row(SC,base,fm,t0)
    except Exception as e:
        print(f"   FAILED: {type(e).__name__}: {str(e)[:200]}")
        b={**base,"status":f"FAILED_{type(e).__name__}","minutes":round((time.time()-t0)/60,1)}
        for k in ["acc","macroF1","macroP","macroR","weightedF1"]: b[k]=float("nan")
        pd.DataFrame([b]).to_csv(SC,mode="a",header=not SC.exists(),index=False)
SIM="cosine"   # 复位
r=pd.read_csv(SC); r=r[r["status"]=="OK"].sort_values("acc",ascending=False)
print("\n"+"="*72); print("similarity 筛选 (冻结探针, swinv2+mpnet, L1_ce, fold0) — 按 acc"); print("="*72)
print(r[["sim","acc","macroF1","weightedF1","macroP","macroR","minutes"]].to_string(index=False))
if len(r): print(f"\n>>> 最优 similarity: {r.iloc[0]['sim']}  acc={r.iloc[0]['acc']:.4f}")

forward 支持 cosine / dot / euclidean
>>> cosine 复用: acc=0.8862
[SKIP] cosine
SIM=dot (frozen, L1_ce)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

[transformers] Swinv2Model LOAD REPORT from: microsoft/swinv2-base-patch4-window8-256
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   unfrozen: vision 0 | text 0
   class_counts: min=3 max=66
   ep000 loss=2.432 val_acc=0.403 val_mF1=0.311
   ep005 loss=0.832 val_acc=0.677 val_mF1=0.530
   ep010 loss=0.609 val_acc=0.677 val_mF1=0.514
   ep015 loss=0.398 val_acc=0.774 val_mF1=0.618
   ep020 loss=0.272 val_acc=0.710 val_mF1=0.532
   ep025 loss=0.398 val_acc=0.758 val_mF1=0.604
   ep030 loss=0.239 val_acc=0.774 val_mF1=0.632
   ep035 loss=0.049 val_acc=0.742 val_mF1=0.564
   ep040 loss=0.158 val_acc=0.774 val_mF1=0.622
   ep045 loss=0.094 val_acc=0.774 val_mF1=0.565
   ep050 loss=0.250 val_acc=0.774 val_mF1=0.533
   ep055 loss=0.376 val_acc=0.758 val_mF1=0.568
   ep060 loss=0.400 val_acc=0.694 val_mF1=0.579
   ep065 loss=0.224 val_acc=0.742 val_mF1=0.610
   ep070 loss=0.196 val_acc=0.790 val_mF1=0.605
   ep075 loss=0.358 val_acc=0.823 val_mF1=0.606
   ep080 loss=0.219 val_acc=0.774 val_mF1=0.633
   ep085 loss=0.297 val_acc=0.855 val_mF1=0.744
   ep090 loss=0.489 val_acc=0.790 val_mF1=0.642
   ep095 loss=0.024 val_acc

Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

[transformers] Swinv2Model LOAD REPORT from: microsoft/swinv2-base-patch4-window8-256
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   unfrozen: vision 0 | text 0
   class_counts: min=3 max=66
   ep000 loss=1.929 val_acc=0.516 val_mF1=0.354
   ep005 loss=0.476 val_acc=0.645 val_mF1=0.528
   ep010 loss=0.447 val_acc=0.758 val_mF1=0.633
   ep015 loss=0.269 val_acc=0.806 val_mF1=0.704
   ep020 loss=0.182 val_acc=0.774 val_mF1=0.696
   ep025 loss=0.341 val_acc=0.806 val_mF1=0.701
   ep030 loss=0.041 val_acc=0.806 val_mF1=0.673
   ep035 loss=0.051 val_acc=0.839 val_mF1=0.739
   ep040 loss=0.030 val_acc=0.806 val_mF1=0.697
   ep045 loss=0.133 val_acc=0.823 val_mF1=0.663
   ep050 loss=0.112 val_acc=0.823 val_mF1=0.701
   ep055 loss=0.046 val_acc=0.790 val_mF1=0.602
   ep060 loss=0.037 val_acc=0.774 val_mF1=0.625
   ep065 loss=0.082 val_acc=0.839 val_mF1=0.695
   ep070 loss=0.139 val_acc=0.806 val_mF1=0.672
   ep075 loss=0.013 val_acc=0.839 val_mF1=0.720
   ep080 loss=0.212 val_acc=0.790 val_mF1=0.656
   ep085 loss=0.069 val_acc=0.823 val_mF1=0.653
   ep090 loss=0.019 val_acc=0.871 val_mF1=0.688
   ep095 loss=0.023 val_acc

In [11]:
# Tier-1 续:冻结探针下筛 loss(swinv2+mpnet, cosine);L1_ce 复用,跑其余 9 个
import time, pandas as pd
pd.set_option("display.float_format", lambda x:f"{x:.4f}")
SIM="cosine"; VIS_ID="microsoft/swinv2-base-patch4-window8-256"; TXT_ID2="sentence-transformers/all-mpnet-base-v2"
FREEZE_VIS, FREEZE_TXT = 1.0, 1.0
LOSS_LIST=["L0_infonce","L1_ce","L2_balanced_softmax","L3_logit_adjust_t10","L4_focal_g2",
           "L5_cb_loss_b099","L6_ldam","L7_supcon","L8_arcface_m05","L9_cosface_m04"]
LC=OUTDIR/"loss_frozen_swinv2_mpnet.csv"
done=set()
if LC.exists(): _d=pd.read_csv(LC); done=set(_d[_d["status"]=="OK"]["loss"])
# L1_ce 复用 encoder 筛选 swinv2+mpnet 那行(frozen+L1_ce+cosine 同配置)
if "L1_ce" not in done and (OUTDIR/"text_screen_frozen.csv").exists():
    vt=pd.read_csv(OUTDIR/"text_screen_frozen.csv")
    fz=vt[(vt["vision"]=="swinv2_base")&(vt["text"]=="mpnet")&(vt["status"]=="OK")]
    if len(fz):
        r0=fz.iloc[0]
        row={"loss":"L1_ce","vision":"swinv2_base","text":"mpnet","sim":"cosine","freeze":1.0,
             "status":"OK","minutes":0.0,"acc":r0["acc"],"macroF1":r0["macroF1"],
             "macroP":r0["macroP"],"macroR":r0["macroR"],"weightedF1":r0["weightedF1"]}
        pd.DataFrame([row]).to_csv(LC,mode="a",header=not LC.exists(),index=False)
        done.add("L1_ce"); print(f">>> L1_ce 复用: acc={r0['acc']:.4f}")
for ln in LOSS_LIST:
    if ln in done: print(f"[SKIP] {ln}"); continue
    FIXED_LOSS=ln; print("="*60); print(f"LOSS={ln} (frozen, cosine)")
    t0=time.time(); base={"loss":ln,"vision":"swinv2_base","text":"mpnet","sim":"cosine","freeze":1.0}
    try: fm=run_combo(VIS_ID,TXT_ID2); append_row(LC,base,fm,t0)
    except Exception as e:
        print(f"   FAILED: {type(e).__name__}: {str(e)[:200]}")
        b={**base,"status":f"FAILED_{type(e).__name__}","minutes":round((time.time()-t0)/60,1)}
        for k in ["acc","macroF1","macroP","macroR","weightedF1"]: b[k]=float("nan")
        pd.DataFrame([b]).to_csv(LC,mode="a",header=not LC.exists(),index=False)
r=pd.read_csv(LC); r=r[r["status"]=="OK"].sort_values("acc",ascending=False)
print("\n"+"="*72); print("loss 筛选 (冻结探针, swinv2+mpnet, cosine, fold0) — 按 acc"); print("="*72)
print(r[["loss","acc","macroF1","weightedF1","macroP","macroR","minutes"]].to_string(index=False))
if len(r): print(f"\n>>> 最优损失(冻结态): {r.iloc[0]['loss']}  acc={r.iloc[0]['acc']:.4f}")

>>> L1_ce 复用: acc=0.8862
LOSS=L0_infonce (frozen, cosine)


Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

[transformers] Swinv2Model LOAD REPORT from: microsoft/swinv2-base-patch4-window8-256
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   unfrozen: vision 0 | text 0
   class_counts: min=3 max=66
   ep000 loss=1.417 val_acc=0.403 val_mF1=0.331
   ep005 loss=0.568 val_acc=0.758 val_mF1=0.562
   ep010 loss=0.347 val_acc=0.790 val_mF1=0.662
   ep015 loss=0.409 val_acc=0.790 val_mF1=0.717
   ep020 loss=0.346 val_acc=0.774 val_mF1=0.641
   ep025 loss=0.398 val_acc=0.790 val_mF1=0.674
   ep030 loss=0.342 val_acc=0.758 val_mF1=0.627
   ep035 loss=0.373 val_acc=0.806 val_mF1=0.643
   ep040 loss=0.332 val_acc=0.839 val_mF1=0.697
   ep045 loss=0.329 val_acc=0.839 val_mF1=0.706
   ep050 loss=0.388 val_acc=0.839 val_mF1=0.654
   ep055 loss=0.321 val_acc=0.806 val_mF1=0.647
   ep060 loss=0.294 val_acc=0.823 val_mF1=0.662
   ep065 loss=0.374 val_acc=0.839 val_mF1=0.667
   early stop @ep69 (best @ep19)
   DONE: acc=0.8130 macroF1=0.7000
[SKIP] L1_ce
LOSS=L2_balanced_softmax (frozen, cosine)


...

   unfrozen: vision 0 | text 0
   class_counts: min=3 max=66
   ep000 loss=22.951 val_acc=0.048 val_mF1=0.004
   ABORT: loss non-finite for 50 batches
LOSS=L9_cosface_m04 (frozen, cosine)


Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

[transformers] Swinv2Model LOAD REPORT from: microsoft/swinv2-base-patch4-window8-256
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   unfrozen: vision 0 | text 0
   class_counts: min=3 max=66
   ep000 loss=14.415 val_acc=0.500 val_mF1=0.402
   ep005 loss=3.233 val_acc=0.726 val_mF1=0.618
   ep010 loss=1.178 val_acc=0.823 val_mF1=0.703
   ep015 loss=0.744 val_acc=0.790 val_mF1=0.673
   ep020 loss=0.326 val_acc=0.790 val_mF1=0.668
   ep025 loss=0.287 val_acc=0.823 val_mF1=0.682
   ep030 loss=0.233 val_acc=0.758 val_mF1=0.626
   ep035 loss=0.186 val_acc=0.806 val_mF1=0.630
   ep040 loss=0.042 val_acc=0.839 val_mF1=0.692
   ep045 loss=0.017 val_acc=0.823 val_mF1=0.680
   ep050 loss=0.379 val_acc=0.855 val_mF1=0.682
   ep055 loss=0.211 val_acc=0.855 val_mF1=0.761
   ep060 loss=0.066 val_acc=0.823 val_mF1=0.647
   ep065 loss=0.148 val_acc=0.806 val_mF1=0.639
   ep070 loss=0.113 val_acc=0.806 val_mF1=0.621
   ep075 loss=0.093 val_acc=0.806 val_mF1=0.637
   ep080 loss=0.193 val_acc=0.774 val_mF1=0.609
   ep085 loss=0.093 val_acc=0.839 val_mF1=0.656
   early stop @ep86 (best @ep56)
   DONE: acc=0.8943 macroF1=0.8284

loss 